# Notebook 1: Capture and Score

Capture wraps an observation with where it came from. Score describes it with three numbers. Neither one decides anything; that is notebook 2's job. There are three TODOs here, a line or two each, with a check cell. Nothing in this notebook writes to the store.

In [1]:
# WHAT THIS CELL DOES: housekeeping. No database calls yet.
#   Finds the repo root and puts it on sys.path so `import memory` works, imports the
#   Weaviate client quietly, and pulls in the pieces this notebook needs: capture,
#   score_candidate, connect, fetch_all. Prints the tier from .env.

# Make `import memory` work whether Jupyter opened the repo root or the notebooks folder.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "memory" / "__init__.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import contextlib
import io

with contextlib.redirect_stderr(io.StringIO()):  # a noisy third-party import warning we do not need in the room
    import weaviate.classes.init  # noqa: F401

from memory import capture, collection_name, connect, describe_tier, fetch_all, score_candidate

print(describe_tier())

WEAVIATE_MODE=docker (tier 1, local Docker container), collection Scott123_MemoryRecord


In [2]:
# WHAT THIS CELL DOES: connects and checks the store is in the expected state.
#   Opens the collection notebook 0 created. It never creates one: if it is missing,
#   the error says to run notebook 0. Counts active records into start_count.
#   0 records: stop and reseed. Not 50: a warning only, scoring still works.
#   start_count is kept so the green check at the end can prove this notebook
#   wrote nothing.
# EXPECT: "store holds 50 active records".

client = connect()
if not client.collections.exists(collection_name()):
    raise RuntimeError("no collection yet. Run notebook 0 first; it creates and seeds the store.")
collection = client.collections.get(collection_name())  # read it; this notebook creates nothing
start_count = len(fetch_all(collection, status="active"))
if start_count == 0:
    raise RuntimeError("no active records. Run the reset cell in notebook 0 to seed the handbook.")
if start_count != 50:
    print(
        f"store holds {start_count} records, not the 50 handbook facts notebook 0 seeds. Scoring still "
        "works; the numbers below may differ a little from the text. For a clean start run the "
        "reset cell in notebook 0."
    )
print(f"store holds {start_count} active records")

store holds 50 active records


## Capture: a candidate is a proposal, not a fact

Before we do anything with an observation, it has to answer five questions: what was observed, who said it to the system, what kind of source that is, where it was observed and how sure the source was. The one that matters most is `source`. It is who said it **to the system**, not who the text claims said it. The code is `memory/capture.py`, a plain [dataclass](https://docs.python.org/3/library/dataclasses.html).

In [3]:
# WHAT THIS CELL DOES: Capture, the first step. Builds one candidate memory.
#   capture() is a dataclass constructor, nothing more. The five provenance fields:
#   text (what), source (who said it TO THE SYSTEM, here the Harbor agent, not the
#   customer the text is about), source_type (human/document/agent/tool/web),
#   context (where it was observed), confidence (the source's own, 0 to 1).
#   topic and scope are routing hints: scope says whose memory this is.
#   as_dict() prints every field including the auto-generated candidate_id and
#   captured_at timestamp. validate() checks the shape of the provenance only.
# EXPECT: the field list, then "problems: none". Nothing is written.

cand = capture(
    text="Halvorsen Dental asked that invoices go to their accounts payable mailbox.",
    source="agent:harbor",            # who said it to the system
    source_type="agent",              # human | document | agent | tool | web
    context="support-session-4471",   # where it was observed
    confidence=0.8,                   # the source's own confidence, 0 to 1
    topic="customer",
    scope="customer:halvorsen-dental",
)
for k, v in cand.as_dict().items():
    print(f"{k:14s} {v}")
print("\nproblems:", cand.validate() or "none")

text           Halvorsen Dental asked that invoices go to their accounts payable mailbox.
source         agent:harbor
source_type    agent
context        support-session-4471
confidence     0.8
topic          customer
scope          customer:halvorsen-dental
tier           session
captured_at    2026-11-19T18:06:27.087366+00:00
candidate_id   daa4442fbc34

problems: none


### TODO 1: fix the provenance

`validate()` checks the shape of a candidate, not whether it is true. This one fails three ways. Fix the three marked fields until it prints `problems: none`.

In [4]:
# WHAT THIS CELL DOES: TODO 1. A candidate with three broken provenance fields.
#   As shipped to attendees: source="forum" (no kind:name shape), context=""
#   (empty), confidence=1.5 (outside 0 to 1). validate() returns one plain
#   sentence per problem and the loop prints them.
#   The fix: source becomes kind:name such as web:supply-forum, context names
#   where it was seen, confidence sits between 0 and 1.
# NOTE: in this completed copy the three fields are already fixed.
# EXPECT: three problems as shipped; "problems: none" once fixed.

sloppy = capture(
    text="Halvorsen Dental asked that invoices go to their accounts payable mailbox.",
    source="user:ada",       # TODO 1: a source is kind:name, like "web:supply-forum"
    source_type="web",
    context="forum_session_1567",           # TODO 1: where was it observed?
    confidence=.86,       # TODO 1: 0 to 1
)
problems = sloppy.validate()
print("problems:", problems or "none")
for problem in problems:
    print("-", problem)

problems: none


## Score: three numbers, no decision

| Score | Question | How it is computed here |
|---|---|---|
| `novelty` | Is this new? | 1 minus the similarity of the nearest active record |
| `utility` | Is it worth keeping? | Specific, confident, right-sized facts score high; hedged fragments score low |
| `contradiction` | Does it disagree with something stored? | A close neighbor on the same topic with a flipped negation or a changed number |

Scoring is one nearest-neighbor query against the store and then some arithmetic. It never writes and it never decides. Weaviate docs: [vector similarity search](https://docs.weaviate.io/weaviate/search/similarity).

In [5]:
# WHAT THIS CELL DOES: Score, the second step, on the Halvorsen candidate.
#   score_candidate() embeds the text, runs ONE near_vector query for the 5 closest
#   active records, then does arithmetic:
#     novelty       = 1 minus the best neighbor similarity
#     utility       = shape of the sentence (length, specifics, hedges, confidence)
#     contradiction = strongest same-topic neighbor with a flipped negation or a
#                     changed number, else 0
#   summary() prints the three numbers; the loop lists the 5 neighbors.
# EXPECT: novelty 0.48, utility 0.96, contradiction 0.00. The neighbors are
#   invoice and customer facts, none closer than 0.52. Nothing is written.

scores = score_candidate(collection, cand)
print(scores.summary())
print("\nnearest active records:")
for n in scores.nearest:
    print(f"  {n.similarity:.3f} [{n.tier}/{n.topic}] {n.text[:70]}")

novelty=0.48 utility=0.96 contradiction=0.00

nearest active records:
  0.517 [org/policy] Vendor invoices are paid on net 30 terms and only against a matching p
  0.491 [org/customer] Halvorsen Dental is a multi-location dental group in Seattle that buys
  0.474 [org/customer] Pinecrest Clinics orders janitorial supplies for 12 sites and prefers 
  0.419 [org/customer] Riverbend School District has a tax-exempt account and requires a purc
  0.354 [org/process] Every refund, credit, and discount is logged in Ledgerstone with the a


### Novelty and duplicates

Give it a sentence the store already holds and novelty drops to zero, with `duplicate_of` pointing at the record. Promote uses that later to corroborate the existing memory instead of storing it twice.

In [6]:
# WHAT THIS CELL DOES: scores a sentence the handbook already holds word for word.
#   The nearest neighbor is at similarity 1.0, so novelty is 0 and duplicate_of
#   points at the stored record (anything at or above 0.92 counts as a duplicate).
# WHY: later, admit() uses duplicate_of to corroborate the existing record instead
#   of storing the same memory twice.
# EXPECT: "novelty=0.00 utility=0.96 contradiction=0.00 | duplicate of: 'Refunds
#   over $500 require approval ...'".

dup = capture(
    text="Refunds over $500 require approval from a support manager before they are issued.",
    source="agent:harbor", source_type="agent", context="support-session-4471",
    confidence=0.8, topic="policy",
)
print(score_candidate(collection, dup).summary())

novelty=0.00 utility=0.96 contradiction=0.00 | duplicate of: 'Refunds over $500 require approval from a support '


### Utility, and TODO 2

`utility_score` rewards the shape of a useful fact, not its truth. The version below is missing one line: a hedged sentence ("I think", "maybe", "might") should lose 0.2. `HEDGES` is the set of hedge words and `tokens` is the set of content words in the text.

In [7]:
# WHAT THIS CELL DOES: TODO 2. A copy of utility_score with one rule to add.
#   The recipe, top to bottom:
#     start at 0.3
#     +0.3 if the sentence has 6 to 40 content words (one fact, not a fragment)
#     +0.2 if it names a number or a proper noun (it is about something specific)
#     -0.2 if it contains a hedge word: think, maybe, might, probably, seems...  (THE TODO)
#     +0.2 times the source's confidence
#     clamp to 0..1
#   content_tokens() lowercases and drops stopwords. HEDGES is the hedge-word set.
#   Two test sentences: the Halvorsen fact and a hedged guess about the warehouse.
# NOTE: in this completed copy the hedge line is already filled in.
# EXPECT: Halvorsen 0.96 either way. The hedged sentence reads 0.76 as shipped
#   and 0.56 once the hedge penalty is in.

import re

from memory.score import HEDGES, content_tokens, numbers_in


def my_utility_score(candidate) -> float:
    text = candidate.text.strip()
    tokens = content_tokens(text)
    score = 0.3
    if 6 <= len(tokens) <= 40:
        score += 0.3  # long enough to mean something, short enough to be one fact
    if numbers_in(text) or re.search(r"\b[A-Z][a-z]+(?: [A-Z][a-z]+)+\b", text):
        score += 0.2  # names a number or a proper noun
    # TODO 2: if any hedge word appears in `tokens`, subtract 0.2
    if tokens & HEDGES:
        score -= 0.2
    score += 0.2 * float(candidate.confidence)
    return round(min(1.0, max(0.0, score)), 4)


hedged = capture(
    text="I think the warehouse might close early sometimes.",
    source="agent:harbor", source_type="agent", context="support-session-4471", confidence=0.8,
)
print(f"{my_utility_score(cand):.2f}  {cand.text}")
print(f"{my_utility_score(hedged):.2f}  {hedged.text}   (0.56 when TODO 2 is done)")

0.96  Halvorsen Dental asked that invoices go to their accounts payable mailbox.
0.56  I think the warehouse might close early sometimes.   (0.56 when TODO 2 is done)


### Contradiction, and TODO 3

For every neighbor that is close enough to be about the same thing, on the same topic and not a duplicate, we ask two cheap questions: did a negation flip, and did a number change? The negation half is written. `cand_nums` holds the candidate's numbers and `numbers_in(n.text)` gives you the neighbor's. A number has changed when both texts carry numbers and the two sets differ.

In [8]:
# WHAT THIS CELL DOES: TODO 3. A copy of contradiction_score with one rule to add.
#   For each of the 5 nearest neighbors:
#     skip if similarity < 0.55 (not about the same thing) or >= 0.92 (a duplicate)
#     skip if the topics differ
#     overlap = shared content words / all content words (Jaccard)
#     flipped_negation: one text has a negation word (not, never, no...) and the other does not
#     changed_number: both texts carry numbers and the sets differ  (THE TODO)
#     if either fired and overlap >= 0.2, that neighbor's similarity is the strength;
#     keep the strongest, and the record it conflicts with
#   Test case: "receipt for items over $2,500" against the handbook's "$250" rule.
# NOTE: in this completed copy the changed_number line is already filled in.
# EXPECT: "contradiction=0.00 against: nothing" as shipped; 0.88 against the
#   itemized-receipt rule once the number check is in.

from memory.score import DUPLICATE_SIMILARITY, NEGATIONS, RELATED_SIMILARITY, nearest_records
from memory.embed import embed_one


def my_contradiction_score(candidate, neighbors):
    cand_tokens = content_tokens(candidate.text)
    cand_neg = bool(cand_tokens & NEGATIONS)
    cand_nums = numbers_in(candidate.text)
    best, best_neighbor = 0.0, None
    for n in neighbors:
        if n.similarity < RELATED_SIMILARITY or n.similarity >= DUPLICATE_SIMILARITY:
            continue  # not about the same thing, or the same memory outright
        if n.topic and candidate.topic and n.topic != candidate.topic:
            continue
        other_tokens = content_tokens(n.text)
        overlap = len(cand_tokens & other_tokens) / max(1, len(cand_tokens | other_tokens))
        flipped_negation = cand_neg != bool(other_tokens & NEGATIONS) # TODO 3: True when both texts carry numbers and they differ
        other_nums = numbers_in(n.text)
        changed_number = bool(cand_nums and other_nums and cand_nums != other_nums)

        if (flipped_negation or changed_number) and overlap >= 0.2:
            if n.similarity > best:
                best, best_neighbor = round(n.similarity, 4), n
    return best, best_neighbor


moved = capture(
    text="Expense reports only need a receipt for items over $2,500.",
    source="agent:harbor", source_type="agent", context="support-session-5110",
    confidence=0.7, topic="policy",
)
neighbors = nearest_records(collection, embed_one(moved.text))
strength, against = my_contradiction_score(moved, neighbors)
print(f"contradiction={strength:.2f}", f"against: {against.text!r}" if against else "against: nothing", "  (0.88 when TODO 3 is done)")

contradiction=0.88 against: 'Expense reports need an itemized receipt for any single item over $250.'   (0.88 when TODO 3 is done)


### Check your work

The reference in `memory/score.py` is what the rest of the workshop runs on. This compares yours against it on four candidates.

In [9]:
# WHAT THIS CELL DOES: the check cell. Yours against the reference (memory/score.py).
#   Four sentences: the Halvorsen fact, the hedged one, the changed-number one, and
#   a neutral systems fact. For each: capture, fetch neighbors once, compute utility
#   and contradiction both ways, flag MISMATCH if either number differs.
# EXPECT: "4 of 4 match the reference" once TODO 2 and TODO 3 are done.

from memory.score import contradiction_score, utility_score

checks = [
    ("Halvorsen Dental asked that invoices go to their accounts payable mailbox.", "customer"),
    ("I think the warehouse might close early sometimes.", "general"),
    ("Expense reports only need a receipt for items over $2,500.", "policy"),
    ("The Portland warehouse added a second loading dock in September.", "systems"),
]


def compare():
    ok = 0
    print(f"{'utility':>15s} {'contradiction':>15s}   text")
    print(f"{'yours / ref':>15s} {'yours / ref':>15s}")
    for text, topic in checks:
        c = capture(text, "agent:harbor", "agent", "support-session-4471", 0.8, topic=topic)
        nbrs = nearest_records(collection, embed_one(text))
        u_mine, u_ref = my_utility_score(c), utility_score(c)
        c_mine, c_ref = my_contradiction_score(c, nbrs)[0], contradiction_score(c, nbrs)[0]
        match = u_mine == u_ref and c_mine == c_ref
        ok += match
        flag = "ok " if match else "MISMATCH"
        print(f"{u_mine:6.2f} / {u_ref:4.2f}   {c_mine:6.2f} / {c_ref:4.2f}   {flag}  {text[:55]}")
    print(f"\n{ok} of {len(checks)} match the reference")
    return ok == len(checks)


compare()

        utility   contradiction   text
    yours / ref     yours / ref
  0.96 / 0.96     0.00 / 0.00   ok   Halvorsen Dental asked that invoices go to their accoun
  0.56 / 0.56     0.00 / 0.00   ok   I think the warehouse might close early sometimes.
  0.96 / 0.96     0.88 / 0.88   ok   Expense reports only need a receipt for items over $2,5
  0.96 / 0.96     0.00 / 0.00   ok   The Portland warehouse added a second loading dock in S

4 of 4 match the reference


True

### What this heuristic misses

Here is a candidate that flatly contradicts the refund rule, with no negation word and no changed number. Watch what the score says.

In [10]:
# WHAT THIS CELL DOES: shows the blind spot of the heuristic.
#   "Refunds over $500 can be approved by any agent" flatly contradicts the handbook
#   (a support manager must approve), but there is no negation word and the number
#   is the same, so neither cheap check fires.
# EXPECT: contradiction=0.00, even though the nearest record is the refund rule at
#   0.82 similarity. Notebook 4 comes back to this exact sentence.

quiet = capture(
    text="Refunds over $500 can be approved by any agent.",
    source="agent:harbor", source_type="agent", context="support-session-4471",
    confidence=0.8, topic="policy",
)
s = score_candidate(collection, quiet)
print(s.summary())
for n in s.nearest[:1]:
    print(f"nearest: {n.similarity:.2f}  {n.text}")

novelty=0.18 utility=0.96 contradiction=0.00
nearest: 0.82  Refunds over $500 require approval from a support manager before they are issued.


## Fell behind, or something broke?

This cell swaps in the finished functions from `memory/score.py` and reruns the check. Notebook 2 uses those anyway, so nothing downstream depends on your TODOs.

In [11]:
# WHAT THIS CELL DOES: the fell-behind cell. Replaces my_utility_score and
#   my_contradiction_score with the finished versions from memory/score.py, then
#   reruns compare(). Safe to run any time: notebook 2 uses the reference code
#   regardless of what happened here.
# EXPECT: "4 of 4 match the reference".

from memory.score import contradiction_score as my_contradiction_score  # noqa: F811
from memory.score import utility_score as my_utility_score  # noqa: F811

assert compare(), "the reference functions should match themselves"

        utility   contradiction   text
    yours / ref     yours / ref
  0.96 / 0.96     0.00 / 0.00   ok   Halvorsen Dental asked that invoices go to their accoun
  0.56 / 0.56     0.00 / 0.00   ok   I think the warehouse might close early sometimes.
  0.96 / 0.96     0.88 / 0.88   ok   Expense reports only need a receipt for items over $2,5
  0.96 / 0.96     0.00 / 0.00   ok   The Portland warehouse added a second loading dock in S

4 of 4 match the reference


## What was Weaviate-specific here, and what was not

Weaviate-specific: `nearest_records` in `memory/score.py`, which is a `near_vector` query with a status filter, about fifteen lines. Portable: the `Candidate` and its validation, all three scores, and the rule that a candidate gets scored against live neighbors before anything decides what happens to it.

## Green check

In [12]:
# WHAT THIS CELL DOES: the green check. Counts active records again and asserts the
#   count equals start_count, proving nothing in this notebook wrote to the store.
#   Closes the connection.

count = len(fetch_all(collection, status="active"))
assert count == start_count, f"started with {start_count} active records, now {count}; this notebook should not have written anything"
print(f"\u2705 {describe_tier()}")
print(f"\u2705 {count} records, same as when we started. Candidate and Scores are ready for the gate in notebook 2.")
client.close()

✅ WEAVIATE_MODE=docker (tier 1, local Docker container), collection Scott123_MemoryRecord
✅ 50 records, same as when we started. Candidate and Scores are ready for the gate in notebook 2.
